<a href="https://colab.research.google.com/github/i0ankurmehta/CertiChain-Blockchain-Project/blob/main/CertiChain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import hashlib
import json
from datetime import datetime

# --- STEP 1: DEFINE THE BLOCK ---
class CertificateBlock:
    def __init__(self, index, student_name, roll_no, course, cgpa, previous_hash):
        self.index = index
        self.timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        self.certificate_data = {
            "student_name": student_name,
            "roll_no": roll_no,
            "course": course,
            "cgpa": cgpa
        }
        self.previous_hash = previous_hash
        self.nonce = 0
        self.hash = self.calculate_hash()

    def calculate_hash(self):
        """Creates a SHA-256 digital fingerprint of the block."""
        block_content = json.dumps({
            "index": self.index,
            "timestamp": self.timestamp,
            "certificate_data": self.certificate_data,
            "previous_hash": self.previous_hash,
            "nonce": self.nonce
        }, sort_keys=True)
        return hashlib.sha256(block_content.encode()).hexdigest()

    def mine_block(self, difficulty):
        """Proof-of-Work: Finds a hash starting with 'difficulty' number of zeros."""
        target = "0" * difficulty
        while self.hash[:difficulty] != target:
            self.nonce += 1
            self.hash = self.calculate_hash()


# --- STEP 2: DEFINE THE BLOCKCHAIN ---
class UniversityLedger:
    def __init__(self):
        self.chain = [self.create_genesis_block()]
        self.difficulty = 3  # Kept at 3 so Colab mines instantaneously

    def create_genesis_block(self):
        return CertificateBlock(0, "GENESIS_BLOCK", "0000", "System Init", 0.0, "0" * 64)

    def issue_certificate(self, student_name, roll_no, course, cgpa):
        prev_block = self.chain[-1]
        new_block = CertificateBlock(
            index=len(self.chain),
            student_name=student_name,
            roll_no=roll_no,
            course=course,
            cgpa=cgpa,
            previous_hash=prev_block.hash
        )
        new_block.mine_block(self.difficulty)
        self.chain.append(new_block)
        print(f"[+] Certificate Mined! Block #{new_block.index} | Student: {student_name} | Hash: {new_block.hash[:16]}...")

    def verify_ledger(self):
        """Checks if any certificate has been altered after issuance."""
        print("\nAuditing University Blockchain...")
        for i in range(1, len(self.chain)):
            current = self.chain[i]
            previous = self.chain[i - 1]

            # Check 1: Did someone edit the data inside the block?
            if current.hash != current.calculate_hash():
                print(f"[ALERT] Tampering detected in Block #{current.index} ({current.certificate_data['student_name']})!")
                print(f"    Stored Hash:     {current.hash[:25]}...")
                print(f"    Calculated Hash: {current.calculate_hash()[:25]}... (MISMATCH!)")
                return False

            # Check 2: Is the chain link broken?
            if current.previous_hash != previous.hash:
                print(f"[ALERT] Broken chain link between Block #{previous.index} and Block #{current.index}!")
                return False

        print("[OK] All certificates are 100% authentic and untampered.")
        return True

    def display_chain(self):
        print("\n" + "="*65)
        print("                UNIVERSITY CERTIFICATE LEDGER")
        print("="*65)
        for b in self.chain[1:]:  # Skip Genesis for cleaner table display
            d = b.certificate_data
            print(f"Block #{b.index} | Roll: {d['roll_no']} | Name: {d['student_name']} | Course: {d['course']} | CGPA: {d['cgpa']}")
            print(f"  ├─ Nonce:     {b.nonce}")
            print(f"  ├─ Prev Hash: {b.previous_hash[:32]}...")
            print(f"  └─ Curr Hash: {b.hash[:32]}...\n")


# --- STEP 3: VIVA DEMONSTRATION ---
# 1. Initialize Ledger & Issue Certificates
uni_chain = UniversityLedger()
uni_chain.issue_certificate("Rohan Sharma", "BCA2401", "BCA Cybersecurity", 8.6)
uni_chain.issue_certificate("Priya Patel", "BCA2402", "BCA Data Science", 9.1)
uni_chain.issue_certificate("Aman Verma", "BCA2403", "BCA Cloud Tech", 7.4)

# 2. Show Valid Ledger
uni_chain.display_chain()
uni_chain.verify_ledger()

# 3. Simulate a Hacker Modifying Grades in the Database
print("\n--- SIMULATING DATABASE ATTACK: Changing Aman's CGPA from 7.4 to 9.8 ---")
uni_chain.chain[3].certificate_data["cgpa"] = 9.8

# 4. Run Audit Again to Catch the Attack
uni_chain.verify_ledger()

[+] Certificate Mined! Block #1 | Student: Rohan Sharma | Hash: 000605803cbcef91...
[+] Certificate Mined! Block #2 | Student: Priya Patel | Hash: 0008e533eb232083...
[+] Certificate Mined! Block #3 | Student: Aman Verma | Hash: 0005c047e6d5eebc...

                UNIVERSITY CERTIFICATE LEDGER
Block #1 | Roll: BCA2401 | Name: Rohan Sharma | Course: BCA Cybersecurity | CGPA: 8.6
  ├─ Nonce:     9219
  ├─ Prev Hash: 149f47ff89a8238fa0906e4b4e60a086...
  └─ Curr Hash: 000605803cbcef91ded2cae46c61acc8...

Block #2 | Roll: BCA2402 | Name: Priya Patel | Course: BCA Data Science | CGPA: 9.1
  ├─ Nonce:     798
  ├─ Prev Hash: 000605803cbcef91ded2cae46c61acc8...
  └─ Curr Hash: 0008e533eb2320834d483060145e6e93...

Block #3 | Roll: BCA2403 | Name: Aman Verma | Course: BCA Cloud Tech | CGPA: 7.4
  ├─ Nonce:     6452
  ├─ Prev Hash: 0008e533eb2320834d483060145e6e93...
  └─ Curr Hash: 0005c047e6d5eebcb0259896bb4de176...


Auditing University Blockchain...
[OK] All certificates are 100% authentic 

False